# Motor Controller Model
## Training a biologically-inspired M1 spiking network with e-prop

This notebook introduces the Motor Controller Model: a recurrent spiking neural network inspired by the **primary motor cortex (M1)** that learns to transform desired reaching trajectories into motor commands.

Rather than focusing on the implementation details of e-prop, this notebook explains the **biological motivation**, the **model architecture**, the **training task**, and the workflow implemented in this repository.

### Learning objectives

After completing this notebook you will be able to:

- Understand the motivation behind the model.
- Configure a motor-learning experiment.
- Visualize the generated planner trajectories.
- Train the recurrent M1 network.
- Interpret the diagnostic figures.
- Evaluate the trained model in inference mode.


# 1. Biological motivation

Reaching movements require transforming a desired movement plan into coordinated motor commands. Inspired by this process, the model is organized into several functional stages:

- A **planner** specifies the desired joint trajectory.
- **Tracking neurons** encode the trajectory as spikes.
- A **Radial Basis (RB) encoder** generates a nonlinear population representation.
- A recurrent spiking network models the dynamics of **M1**.
- Two readout populations encode positive and negative motor commands.

The recurrent network is trained using **Eligibility Propagation (e-prop)**, an online learning rule that combines local eligibility traces with neuron-specific learning signals, providing a biologically plausible alternative to Backpropagation Through Time (BPTT) [1,2].


# 2. Model architecture

> **TODO:** Replace the diagram below with the project schematic.

```text
Desired trajectory
        │
        ▼
Tracking neurons ──► RB encoder ──► Recurrent M1 network ──► Readout neurons ──► Motor command
```

The planner provides the desired movement, the RB layer expands the representation into a nonlinear feature space, the recurrent network learns the temporal dynamics, and the readout neurons decode the resulting motor command.


# 3. Learning with e-prop

Learning is performed with **Eligibility Propagation (e-prop)** [1], while this implementation relies on the event-driven implementation available in NEST [2].

From the perspective of this project, e-prop is simply the mechanism that allows the recurrent network to adapt its plastic synapses during repeated reaching movements. The details of the learning rule are encapsulated by the neuron and synapse models, allowing users to focus on the motor-learning task rather than the underlying optimization algorithm.


# 4. Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from motor_controller_model.config_schema import (
    MotorControllerConfig,
)
from motor_controller_model.signals import generate_training_signals
from motor_controller_model.m1_factory import get_m1_or_train
from motor_controller_model.run_m1 import run_inference_test


# 5. Configure an experiment

All experiment parameters are grouped in a single `MotorControllerConfig`. This includes the network architecture, simulation settings, training task, recording options, and plotting configuration.


In [ ]:
config = MotorControllerConfig.from_yaml("experiments/epropplus_poisson_background/config.yaml")

# Small tutorial experiment
config.task.n_iter = 100
config.plotting.do_plotting = True

config


# 6. Visualize the training task

Before training, it is useful to inspect the planner trajectories and the corresponding target output signals. This confirms that the generated reaching movements match the intended experiment.


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(6, 4), sharex=True)

offset = 0

for spec in config.training.trajectories:
	signals = generate_training_signals(
		spec,
		config.training,
		config.simulation.step,
		config.task.input_shift_ms,
	)

	t = np.arange(len(signals.input_trajectory)) + offset

	ax[0].plot(t, signals.input_trajectory, "k", lw=2)
	ax[1].plot(t, signals.target_rates_pos, color="tab:blue")
	ax[1].plot(t, signals.target_rates_neg, color="tab:red")

	if offset:
		for a in ax:
			a.axvline(offset, color="0.85", ls="--", lw=0.8)

	offset += len(t)

ax[0].set(title="Desired trajectories", ylabel="Angle (rad)")
ax[1].set(
	title="Target motor commands",
	xlabel="Simulation step",
	ylabel="Target rate (Hz)",
)

ax[1].plot([], [], color="tab:red", label="Positive")
ax[1].plot([], [], color="tab:blue", label="Negative")
ax[1].legend(frameon=False)

plt.tight_layout()
plt.show()

# 7. Train the network

Training automatically:

1. Builds the M1 network.
2. Generates the planner inputs.
3. Runs e-prop optimization.
4. Stores the learned synaptic weights.
5. Produces diagnostic figures for later inspection.


In [ ]:
artifacts_dir = Path("tutorial_results")
artifacts_dir.mkdir(exist_ok=True)

network, outputs = get_m1_or_train(
    config=config,
    artifacts_dir=artifacts_dir,
    nest_module="motor_neuron_module",
    force_retrain=True,
    return_outputs=True,
)


# 8. Inspect the training diagnostics

The training pipeline automatically generates several figures.

When reviewing them, consider the following questions:

- Does the training loss converge?
- Is the recurrent activity stable throughout learning?
- How do the recurrent and readout weights evolve?
- Does the learned connectivity appear structured?


In [ ]:
from motor_controller_model.plot_results import tutorial_plot_loss_curve

tutorial_plot_loss_curve(outputs.loss)

# 9. Run inference

After training, plasticity is disabled and the learned weights are reused to evaluate the network on the configured reaching trajectories.


In [ ]:
readout_pos, readout_neg = run_inference_test(
    config=config,
    network=network,
    artifacts_dir=artifacts_dir,
    nest_module="motor_neuron_module",
)

img = artifacts_dir / "standalone_inference_test.png"
if img.exists():
    plt.figure(figsize=(12,8))
    plt.imshow(Image.open(img))
    plt.axis("off")
    plt.show()


# 10. Exploring the model

Some interesting directions to explore are:

- Changing the number of recurrent neurons.
- Adding new reaching trajectories.
- Modifying the planner gains.
- Enabling or tuning background activity.
- Comparing different optimization settings.

Each experiment stores its configuration alongside the learned weights and generated figures, making results reproducible.


# References

**[1]** Bellec, G., Scherr, F., Subramoney, A., *et al.* (2020). *A solution to the learning dilemma for recurrent networks of spiking neurons*. **Nature Communications**, 11, 3625.

**[2]** Espinoza Valverde, J. A., *et al.* (2025). *Event-driven Eligibility Propagation in NEST*. Manuscript describing the event-driven implementation of e-prop used by this project.

**[3]** Neftci, E. O., Mostafa, H., & Zenke, F. (2019). *Surrogate Gradient Learning in Spiking Neural Networks*. **IEEE Signal Processing Magazine**, 36(6), 51–63.
